# 🏆 Amazon ML Challenge 2026 — Business Entity Resolution
## Single Master Pipeline Notebook (v3 — Production Grade)

| Setting | Value |
|---|---|
| Target Metric | Macro-Averaged F₀.₅ |
| Scale | 2.2M S1 · 5M S2 · 5.3M S3 (train); 1.7M S1 (test) |
| Open-Set | France appears **only** in test — never hardcode country |
| Outputs | `output/matching_results.tsv` + `output/candidate_pairs.tsv` |

> **Run top-to-bottom.** Every cell is self-contained; no external `.py` files needed.
---

### ⚙️ Cell 1 · Environment Setup & Drive Mount

In [ ]:
import os, sys, gc, re, time, logging, unicodedata, subprocess, pickle, shutil
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Set, Optional

# ── Google Colab detection & Drive mount ────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Install pinned deps (T4-compatible) ──────────────────────
!pip install -q lightgbm>=4.3.0 scikit-learn>=1.4.0 rapidfuzz>=3.8.0 joblib>=1.4.0 psutil cupy-cuda12x jellyfish catboost xgboost sentence-transformers networkx

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
import joblib
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
import psutil
try:
    import jellyfish
    HAS_JELLYFISH = True
except ImportError:
    HAS_JELLYFISH = False
    print('jellyfish not found — phonetic blockers will use Soundex fallback')

# ── CuPy GPU setup (T4 safe, CPU fallback) ──────────────────
try:
    import cupy as cp
    _ = cp.array([1.0])  # trigger CUDA context init
    free_vram, total_vram = cp.cuda.runtime.memGetInfo()
    GPU_NAME = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    print(f'GPU  : {GPU_NAME}')
    print(f'VRAM : {free_vram/1e9:.1f} GB free / {total_vram/1e9:.1f} GB total')
    USE_GPU = True
except Exception as e:
    print(f'GPU not available ({e}) — using CPU path')
    cp = None
    USE_GPU = False

# ── Root paths ───────────────────────────────────────────────
if IN_COLAB:
    DRIVE_ROOT    = Path('/content/drive/MyDrive/Amazon-ML-entity-resolution')
    DRIVE_DATASET = DRIVE_ROOT / 'dataset'
    LOCAL_DATASET = Path('/content/dataset')
    if DRIVE_DATASET.exists() and not LOCAL_DATASET.exists():
        print('Copying dataset Drive -> local SSD ...')
        shutil.copytree(DRIVE_DATASET, LOCAL_DATASET, dirs_exist_ok=True)
        print('Dataset ready on local SSD.')
    ROOT = Path('/content')
else:
    DRIVE_ROOT = None
    ROOT = Path('.').resolve()

TRAIN_DIR  = ROOT / 'dataset' / 'train'
TEST_DIR   = ROOT / 'dataset' / 'test'
OUTPUT_DIR = ROOT / 'output'
MODEL_DIR  = ROOT / 'models'
CKPT_DIR   = ROOT / 'checkpoints'
REPORT_DIR = ROOT / 'reports' / 'error_analysis'

for _d in [OUTPUT_DIR, MODEL_DIR, REPORT_DIR,
           CKPT_DIR/'preprocess', CKPT_DIR/'blocking', CKPT_DIR/'features']:
    _d.mkdir(parents=True, exist_ok=True)

# Mirror checkpoint dir on Drive so we survive session resets
if DRIVE_ROOT:
    DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

def save_to_drive(local_path: Path):
    """Copy a local checkpoint file to Drive for persistence."""
    if DRIVE_ROOT:
        dest = DRIVE_ROOT / 'checkpoints' / local_path.name
        shutil.copy(local_path, dest)
        print(f'  Backed up to Drive: {dest}')

def load_from_drive(filename: str) -> Optional[Path]:
    """If local checkpoint missing, try loading from Drive."""
    local = CKPT_DIR / filename
    if local.exists(): return local
    if DRIVE_ROOT:
        remote = DRIVE_ROOT / 'checkpoints' / filename
        if remote.exists():
            shutil.copy(remote, local)
            print(f'  Restored from Drive: {filename}')
            return local
    return None

def free_gpu():
    """Free GPU memory pool between heavy operations."""
    if USE_GPU:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    gc.collect()

def report_memory():
    ram_free = psutil.virtual_memory().available / 1e9
    msg = f'RAM free: {ram_free:.1f} GB'
    if USE_GPU:
        free_v, total_v = cp.cuda.runtime.memGetInfo()
        msg += f'  |  VRAM free: {free_v/1e9:.1f}/{total_v/1e9:.1f} GB'
    print(msg)

# ── Diagnostics ─────────────────────────────────────────────
ram_gb = psutil.virtual_memory().total / 1e9
print(f'System RAM : {ram_gb:.1f} GB  |  CPUs: {os.cpu_count()}')
report_memory()
print('Environment ready.')


### ⚙️ Cell 2 · Central Configuration — Edit Before Each Run

In [ ]:
# ===========================================================
# EXPERIMENT SETTINGS — change here, nowhere else
# ===========================================================
SAMPLE_S1      = 50_000   # int or None for full 2.2M run
FULL_S23       = True     # True = full S2/S3 (Phase 2 gate), False = reduced dev
K_CANDIDATES   = 50       # top-K per S1 entity
RANDOM_STATE   = 42

# ── TF-IDF Vectoriser ────────────────────────────────────────
TFIDF_ANALYZER     = 'char_wb'
TFIDF_NGRAM_RANGE  = (2, 4)
TFIDF_MAX_FEATURES = 50_000
TFIDF_MIN_DF       = 2

# ── T4 GPU Memory Safety (tune if OOM) ──────────────────────
# Rule: S1 query matrix stays dense (S1_BATCH_SIZE x vocab) in GPU VRAM.
# S23 stays as CSR sparse on CPU; dot product via scipy (avoids 80 GB dense alloc).
# S1 dense batch: 300 x 50000 x 4 bytes = 60 MB  -> safe
# S23 sparse CSR (10M x 50K, ~2% density): ~4 GB  -> safe on CPU RAM
S1_BATCH_SIZE  = 300       # S1 rows per cosine search batch
S23_CHUNK_SIZE = 400_000   # S23 rows per chunk if chunking needed
MIN_SIM_SCORE  = 0.05

# ── 3-way split ─────────────────────────────────────────────
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# Holdout = remaining 0.15 — check ONCE, then freeze model

# ── LightGBM (CPU, not GPU — more stable on Colab T4) ───────
LGBM_PARAMS = dict(
    objective        = 'binary',
    metric           = 'auc',
    num_leaves       = 127,
    learning_rate    = 0.05,
    n_estimators     = 1500,
    min_child_samples= 20,
    subsample        = 0.80,
    colsample_bytree = 0.80,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    n_jobs           = -1,           # use all CPU cores
    random_state     = RANDOM_STATE,
    verbose          = -1,
)
EARLY_STOPPING_ROUNDS = 50
THRESHOLD_GRID = [round(x * 0.05, 2) for x in range(4, 20)]  # 0.20 -> 0.95

print(f'Config loaded: SAMPLE_S1={SAMPLE_S1}  FULL_S23={FULL_S23}  K={K_CANDIDATES}  USE_GPU={USE_GPU}')
report_memory()

# ── Ensemble Thresholds & Singleton Shield ───────────────────
T_MATCH   = 0.55   # primary match threshold (swept on validation)
T_HIGH    = 0.75   # high-confidence threshold for triangulation
T_SINGLE  = 0.65   # singleton guardrail: lone marginal predictions are pruned

# ── Ensemble Weights (geometric mean: LGBM^0.5 * CatBoost^0.3 * XGB^0.2) ─
ENSEMBLE_WEIGHTS = {'lgbm': 0.5, 'catboost': 0.3, 'xgb': 0.2}

# ── Multilingual Embeddings (paraphrase-multilingual-MiniLM-L12-v2) ─────
USE_EMBEDDINGS = True     # set False to skip embedding feature (saves ~3min)
EMBED_BATCH_SIZE = 512    # sentences per embedding batch
EMBED_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f'Ensemble weights: {ENSEMBLE_WEIGHTS}')
print(f'Singleton guardrail threshold: T_SINGLE={T_SINGLE}')
print(f'Triangulation high-threshold: T_HIGH={T_HIGH}')


### 🔤 Cell 3 · Text Normalisation Module

In [ ]:
# ===========================================================
# MODULE: Preprocessing (NFKC + full legal suffix expansion)
# Covers: French (SAS/SARL/SASU/EURL/SCI/GIE), Indian (PVT LTD/LLP/M/S), US (LLC/INC)
# ===========================================================
_NAME_ABBREVS_RAW = {
    # US / International legal suffixes
    r'\bcorp\b':'corporation', r'\bco\b':'company', r'\binc\b':'incorporated',
    r'\bltd\b':'limited', r'\bllc\b':'limited liability company',
    r'\bllp\b':'limited liability partnership', r'\bpvt\b':'private',
    r'\bpvt ltd\b':'private limited', r'\bprivate limited\b':'private limited',
    r'\bpte\b':'private', r'\bpte ltd\b':'private limited',
    r'\blp\b':'limited partnership', r'\bplc\b':'public limited company',
    # French legal suffixes (test set contains France - open set)
    r'\bsas\b':'societe par actions simplifiee',
    r'\bsasu\b':'societe par actions simplifiee unipersonnelle',
    r'\bsarl\b':'societe a responsabilite limitee',
    r'\beurl\b':'entreprise unipersonnelle a responsabilite limitee',
    r'\bselarl\b':'societe exercice liberal a responsabilite limitee',
    r'\bsci\b':'societe civile immobiliere',
    r'\bgie\b':'groupement interet economique',
    r'\bsa\b':'societe anonyme',
    r'\bsnc\b':'societe en nom collectif',
    # Indian trade suffixes
    r'\bm/s\b':'messrs', r'\bms\b':'messrs',
    r'\bprop\b':'proprietor', r'\bproprietorship\b':'proprietor',
    # Common business words
    r'\b&\b':'and', r'\bsvc\b':'services', r'\bsvcs\b':'services',
    r'\btech\b':'technology', r'\bintl\b':'international', r'\bmfg\b':'manufacturing',
    r'\bmgmt\b':'management', r'\bassoc\b':'associates', r'\bnatl\b':'national',
    r'\bgrp\b':'group', r'\bent\b':'enterprises', r'\bhldgs\b':'holdings',
    r'\binds\b':'industries', r'\bsol\b':'solutions', r'\bsolns\b':'solutions',
    r'\bmkt\b':'marketing', r'\bconsult\b':'consulting', r'\bdev\b':'development',
    r'\bfinl\b':'financial', r'\bhosp\b':'hospital', r'\bmed\b':'medical',
    r'\bpharm\b':'pharmaceutical',
}
_ADDR_ABBREVS_RAW = {
    r'\brd\b':'road', r'\bst\b':'street', r'\bave\b':'avenue',
    r'\bblvd\b':'boulevard', r'\bdr\b':'drive', r'\bln\b':'lane',
    r'\bct\b':'court', r'\bpl\b':'place', r'\bhwy\b':'highway',
    r'\bpkwy\b':'parkway', r'\bexpy\b':'expressway', r'\bste\b':'suite',
    r'\bapt\b':'apartment', r'\bflr\b':'floor', r'\bbldg\b':'building',
    r'\bnw\b':'northwest', r'\bne\b':'northeast',
    r'\bsw\b':'southwest', r'\bse\b':'southeast',
    # Indian address abbreviations
    r'\bnr\b':'near', r'\bopp\b':'opposite', r'\bsoc\b':'society',
    r'\bchs\b':'cooperative housing society', r'\bappt\b':'apartment',
    # French address abbreviations
    r'\bav\b':'avenue', r'\bbd\b':'boulevard', r'\bpl\b':'place',
    r'\bche\b':'chemin', r'\bres\b':'residence', r'\bbt\b':'batiment',
}

def _compile(raw): return [(re.compile(p, re.IGNORECASE), r) for p, r in raw.items()]
_NAME_ABBREVS = _compile(_NAME_ABBREVS_RAW)
_ADDR_ABBREVS = _compile(_ADDR_ABBREVS_RAW)
_PUNCT_RE  = re.compile(r'[^\w\s]')
_SPACE_RE  = re.compile(r'\s+')
_DIGIT_RE  = re.compile(r'\d+')
_POSTAL_RE = re.compile(r'\b(\d{5,6}|[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\b')  # PIN/ZIP/UK postcode
_HOUSE_RE  = re.compile(r'^\d+[A-Za-z]?\b')  # leading house number

def _safe_str(v) -> str:
    if v is None: return ''
    s = str(v).strip()
    return '' if s.lower() in ('nan', 'none', '') else s

def _apply(text, abbrevs):
    for pat, rep in abbrevs:
        text = pat.sub(rep, text)
    return text

def normalize_name(text) -> str:
    t = unicodedata.normalize('NFKC', _safe_str(text)).lower()
    t = _SPACE_RE.sub(' ', _PUNCT_RE.sub(' ', t)).strip()
    t = _SPACE_RE.sub(' ', _apply(t, _NAME_ABBREVS)).strip()
    return t

def normalize_address(text) -> str:
    t = unicodedata.normalize('NFKC', _safe_str(text)).lower()
    t = _SPACE_RE.sub(' ', _PUNCT_RE.sub(' ', t)).strip()
    t = _SPACE_RE.sub(' ', _apply(t, _ADDR_ABBREVS)).strip()
    return t

def extract_postal_code(text: str) -> str:
    """Extract first PIN/ZIP/postcode from raw address string."""
    m = _POSTAL_RE.search(_safe_str(text).upper())
    return m.group(0).replace(' ', '') if m else ''

def extract_house_number(text: str) -> str:
    """Extract leading house number (e.g. '42B' from '42B Main Street')."""
    t = _safe_str(text).strip()
    m = _HOUSE_RE.match(t)
    return m.group(0).lower() if m else ''

def phonetic_key(text: str) -> str:
    """Double Metaphone key (jellyfish) with Soundex fallback."""
    if not text: return ''
    if HAS_JELLYFISH:
        try:
            # Use first token's DM primary key
            toks = text.split()[:4]
            return ' '.join(jellyfish.soundex(t) for t in toks if t)
        except Exception:
            pass
    # Soundex fallback (pure Python)
    def _soundex(s):
        s = s.upper().strip()
        if not s: return ''
        table = str.maketrans('BFPVCGJKQSXZDTLMNR', '111122222222334556')
        coded = s[0] + s[1:].translate(table).replace('0','')
        deduped = coded[0] + ''.join(c for i, c in enumerate(coded[1:], 1) if c != coded[i-1])
        return (deduped + '000')[:4]
    toks = text.split()[:4]
    return ' '.join(_soundex(t) for t in toks if t)

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ('business_name', 'business_address', 'country'):
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str).str.strip()
            df.loc[df[col].str.lower().isin(['nan', 'none']), col] = ''
    df['name_norm']     = df['business_name'].apply(normalize_name)
    df['addr_norm']     = df['business_address'].apply(normalize_address)
    df['country_norm']  = df['country'].apply(lambda x: _safe_str(x).lower().strip())
    df['addr_nums']     = df['business_address'].apply(
        lambda x: ' '.join(_DIGIT_RE.findall(_safe_str(x))))
    df['postal_code']   = df['business_address'].apply(extract_postal_code)
    df['house_num']     = df['business_address'].apply(extract_house_number)
    df['phonetic_name'] = df['name_norm'].apply(phonetic_key)
    df['blocking_text'] = df['name_norm']  # primary TF-IDF input
    return df

print('Preprocessing module loaded (French/Indian/US suffixes + postal + phonetic).')


### 📂 Cell 4 · Data Loading with Checkpointing

In [ ]:
def load_data(split='train', sample_s1=None, full_s23=False):
    """
    Load and preprocess train or test data.
    - sample_s1:  int → sample this many S1 rows; None → use all.
    - full_s23:   False → co-sample S2/S3 to match S1 GT + extras.
                  True  → keep full S2/S3 universe (realistic stress-test).
    """
    ckpt = CKPT_DIR / 'preprocess' / f'{split}_s1{sample_s1}_fs23{full_s23}.pkl'
    if ckpt.exists():
        print(f'  ✔ Loading from checkpoint: {ckpt}')
        return joblib.load(ckpt)

    d = TRAIN_DIR if split == 'train' else TEST_DIR
    s1_df = pd.read_csv(d/f'{split}_source1.tsv', sep='\t', dtype=str, keep_default_na=False)
    s2_df = pd.read_csv(d/f'{split}_source2.tsv', sep='\t', dtype=str, keep_default_na=False)
    s3_df = pd.read_csv(d/f'{split}_source3.tsv', sep='\t', dtype=str, keep_default_na=False)
    gt_df = pd.read_csv(d/'train_ground_truth.tsv', sep='\t', dtype=str, keep_default_na=False) if split=='train' else None

    if sample_s1 and split == 'train':
        s1_df = s1_df.sample(n=min(sample_s1, len(s1_df)), random_state=RANDOM_STATE).reset_index(drop=True)

    if not full_s23 and sample_s1 and split == 'train' and gt_df is not None:
        # Reduced-dev: keep GT S2/S3 records + some extras
        s1_ids = set(s1_df['entity_id'])
        gt_sub = gt_df[gt_df['source1_entity_id'].isin(s1_ids)]
        gt_match_ids = set()
        for _, r in gt_sub.iterrows():
            m = r.get('matched_entity_ids', '').strip()
            if m: gt_match_ids.update(m.split(','))
        rng = np.random.default_rng(RANDOM_STATE)
        def _sub(df, ids, n_extra):
            gt_rows = df[df['entity_id'].isin(ids)]
            rest = df[~df['entity_id'].isin(ids)]
            take = min(n_extra, len(rest))
            extra = rest.iloc[rng.choice(len(rest), take, replace=False)]
            return pd.concat([gt_rows, extra], ignore_index=True)
        s2_gt = {x for x in gt_match_ids if x.startswith('S2-')}
        s3_gt = {x for x in gt_match_ids if x.startswith('S3-')}
        s2_df = _sub(s2_df, s2_gt, sample_s1 * 5)
        s3_df = _sub(s3_df, s3_gt, sample_s1 * 5)
        print(f'  Dev pool: S2={len(s2_df):,}  S3={len(s3_df):,}')

    print(f'  Preprocessing S1={len(s1_df):,}  S2={len(s2_df):,}  S3={len(s3_df):,} …')
    s1 = preprocess_df(s1_df)
    s2 = preprocess_df(s2_df)
    s3 = preprocess_df(s3_df)

    gt_dict = {}
    if gt_df is not None:
        s1_set = set(s1['entity_id'])
        for _, r in gt_df.iterrows():
            sid = r['source1_entity_id']
            if sid in s1_set:
                m = r.get('matched_entity_ids', '').strip()
                gt_dict[sid] = [x for x in m.split(',') if x] if m else []

    result = (s1, s2, s3, gt_dict)
    joblib.dump(result, ckpt)
    print(f'  ✔ Checkpoint saved: {ckpt}')
    return result

print('✅ Data loader ready.')


### 🔍 Cell 5 · Multi-Blocker Candidate Generation

**6 independent blocking routes (union → dedup → top-K):**
| Blocker | Method | Good For |
|---|---|---|
| A | Name char-TF-IDF cosine | Baseline, handles most cases |
| B | Address char-TF-IDF cosine | Different names, same address |
| C | Exact normalised name | Free, catches perfect normalisation |
| D | Rare-token inverted index | Unique brand / product names |
| E | Numeric / PIN overlap | Street numbers, postal codes |
| F | Full-doc TF-IDF | Name+addr merged vector |

In [ ]:
# ===========================================================
# MODULE: Multi-Blocker Candidate Generation (8 independent routes)
# A: Name TF-IDF  B: Addr TF-IDF  C: Exact name  D: Rare token
# E: Numeric      F: Full-doc     G: Phonetic     H: Postal+House (DBA)
# ===========================================================
logger = logging.getLogger('blocking')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-7s | %(message)s', datefmt='%H:%M:%S')

def _build_tfidf(texts: List[str]):
    vec = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER, ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES, min_df=TFIDF_MIN_DF,
        sublinear_tf=True, strip_accents=None, norm='l2', dtype=np.float32,
    )
    return vec, vec.fit_transform(texts)

def _cosine_topk_cpu(q_mat, s23_mat, s23_ids, k):
    """Memory-safe sparse-dense cosine search (CPU). Avoids T4 OOM."""
    results = defaultdict(list)
    n_q = q_mat.shape[0]
    for start in range(0, n_q, S1_BATCH_SIZE):
        batch = q_mat[start:start+S1_BATCH_SIZE].toarray().astype(np.float32)  # (batch, vocab)
        # S23 stays sparse on CPU — dot via scipy to avoid dense alloc of full S23
        scores = (s23_mat @ batch.T).toarray()  # (N_s23, batch)
        for bi in range(scores.shape[1]):
            row = scores[:, bi]
            top_idx = np.argpartition(row, -min(k, len(row)))[-k:]
            for idx in top_idx:
                if row[idx] >= MIN_SIM_SCORE:
                    results[start+bi].append((s23_ids[idx], float(row[idx])))
    return results

def _inverted_index(df, key_col):
    idx = defaultdict(list)
    for eid, key in zip(df['entity_id'], df[key_col]):
        if key: idx[key].append(eid)
    return idx

def _rare_token_index(df, col, max_df=10):
    freq = defaultdict(int)
    for text in df[col]:
        for tok in text.split():
            if len(tok) >= 3: freq[tok] += 1
    rare = {t for t, c in freq.items() if 2 <= c <= max_df}
    idx = defaultdict(set)
    for eid, text in zip(df['entity_id'], df[col]):
        for tok in text.split():
            if tok in rare: idx[tok].add(eid)
    return idx

def _num_overlap_index(df):
    idx = defaultdict(set)
    for eid, nums in zip(df['entity_id'], df['addr_nums']):
        for n in nums.split():
            if len(n) >= 3: idx[n].add(eid)
    return idx

def _phonetic_index(df):
    """
    Blocker G: Double Metaphone inverted index (per token).
    Uses jellyfish.metaphone (primary key) per token; Soundex fallback.
    Catches transliteration: 'Sharma' -> 'XRMA', 'Sarma' -> 'XRMA'.
    """
    idx = defaultdict(set)
    for eid, text in zip(df['entity_id'], df['phonetic_name']):
        # phonetic_name is already space-joined metaphone/soundex keys
        for key in text.split():
            if key: idx[key].add(eid)
    # Also index individual token-level DM keys for richer recall
    if HAS_JELLYFISH:
        for eid, name in zip(df['entity_id'], df['name_norm']):
            for tok in name.split():
                if len(tok) >= 3:
                    pk = jellyfish.metaphone(tok)
                    if pk: idx[pk].add(eid)
    return idx

def _postal_house_index(df):
    """
    Blocker H: Country-keyed (postal_code + house_number) inverted index.
    Prevents cross-country false collisions (e.g. US 90210 ≠ India 90210).
    Catches DBA/trade-name pairs at the identical physical address.
    """
    idx = defaultdict(set)
    for eid, postal, house, country in zip(
            df['entity_id'], df['postal_code'], df['house_num'], df['country_norm']):
        country_prefix = (country[:2] if country else 'XX').upper()
        if postal and house:
            key = f'{country_prefix}|{postal}|{house}'
            idx[key].add(eid)
        if postal and len(postal) >= 5:
            idx[f'{country_prefix}|{postal}'].add(eid)
    # Also build digit-set key (legacy Blocker H logic) for addresses w/o postal extraction
    for eid, addr_nums, country in zip(df['entity_id'], df['addr_nums'], df['country_norm']):
        country_prefix = (country[:2] if country else 'XX').upper()
        digits = [d for d in addr_nums.split() if len(d) >= 3]
        if digits:
            key = f'{country_prefix}_' + '_'.join(sorted(digits))
            idx[key].add(eid)
    return idx

def generate_candidates(s1_df, s2_df, s3_df, k=None, gt_dict=None):
    k = k or K_CANDIDATES
    t0 = time.time()
    s1_ids = list(s1_df['entity_id'])
    cands_raw: Dict[str, List[Tuple[str,float]]] = {sid: [] for sid in s1_ids}
    s23_df = pd.concat([s2_df, s3_df], ignore_index=True)
    s23_ids = list(s23_df['entity_id'])

    # ── Blocker A: Name TF-IDF ───────────────────────────────
    logger.info('Blocker A: Name TF-IDF ...')
    all_texts = list(s1_df['blocking_text']) + list(s23_df['blocking_text'])
    vec_n, _ = _build_tfidf(all_texts)
    s1_mat  = vec_n.transform(list(s1_df['blocking_text']))
    s23_mat = vec_n.transform(list(s23_df['blocking_text']))
    hits = _cosine_topk_cpu(s1_mat, s23_mat, s23_ids, k)
    for i, sid in enumerate(s1_ids):
        for cid, sc in hits.get(i, []): cands_raw[sid].append((cid, sc))
    del s1_mat, s23_mat, hits; free_gpu()

    # ── Blocker B: Address TF-IDF ────────────────────────────
    logger.info('Blocker B: Address TF-IDF ...')
    addr_texts = list(s1_df['addr_norm']) + list(s23_df['addr_norm'])
    if sum(bool(t.strip()) for t in addr_texts) > 200:
        vec_a, _ = _build_tfidf(addr_texts)
        s1_amat  = vec_a.transform(list(s1_df['addr_norm']))
        s23_amat = vec_a.transform(list(s23_df['addr_norm']))
        hits_b = _cosine_topk_cpu(s1_amat, s23_amat, s23_ids, k//2)
        for i, sid in enumerate(s1_ids):
            for cid, sc in hits_b.get(i, []): cands_raw[sid].append((cid, sc*0.8))
        del s1_amat, s23_amat, hits_b; free_gpu()

    # ── Blocker C: Exact normalised name ─────────────────────
    logger.info('Blocker C: Exact name ...')
    exact_idx = _inverted_index(s23_df, 'name_norm')
    for sid, name in zip(s1_df['entity_id'], s1_df['name_norm']):
        if name:
            for cid in exact_idx.get(name, [])[:20]: cands_raw[sid].append((cid, 1.5))

    # ── Blocker D: Rare-token index ───────────────────────────
    logger.info('Blocker D: Rare token ...')
    rare_idx = _rare_token_index(s23_df, 'name_norm', max_df=10)
    for sid, name in zip(s1_df['entity_id'], s1_df['name_norm']):
        hits_d = set()
        for tok in name.split():
            if len(tok) >= 3:
                hits_d.update(rare_idx.get(tok, set()))
        for cid in list(hits_d)[:15]: cands_raw[sid].append((cid, 0.70))

    # ── Blocker E: Numeric / PIN overlap ─────────────────────
    logger.info('Blocker E: Numeric overlap ...')
    num_idx = _num_overlap_index(s23_df)
    for sid, nums in zip(s1_df['entity_id'], s1_df['addr_nums']):
        hits_e = set()
        for n in nums.split():
            if len(n) >= 3: hits_e.update(num_idx.get(n, set()))
        for cid in list(hits_e)[:10]: cands_raw[sid].append((cid, 0.60))

    # ── Blocker F: Full-doc TF-IDF (name + addr) ─────────────
    logger.info('Blocker F: Full-doc TF-IDF ...')
    doc_texts_s1  = [f"{n} {a}" for n, a in zip(s1_df['name_norm'],  s1_df['addr_norm'])]
    doc_texts_s23 = [f"{n} {a}" for n, a in zip(s23_df['name_norm'], s23_df['addr_norm'])]
    vec_f, _ = _build_tfidf(doc_texts_s1 + doc_texts_s23)
    s1_fmat  = vec_f.transform(doc_texts_s1)
    s23_fmat = vec_f.transform(doc_texts_s23)
    hits_f = _cosine_topk_cpu(s1_fmat, s23_fmat, s23_ids, k//3)
    for i, sid in enumerate(s1_ids):
        for cid, sc in hits_f.get(i, []): cands_raw[sid].append((cid, sc*0.75))
    del s1_fmat, s23_fmat, hits_f; free_gpu()

    # ── Blocker G: Phonetic inverted index (NEW) ──────────────
    logger.info('Blocker G: Phonetic name index ...')
    phonetic_idx = _phonetic_index(s23_df)
    for sid, pkey in zip(s1_df['entity_id'], s1_df['phonetic_name']):
        if pkey:
            hits_g = phonetic_idx.get(pkey, set())
            for cid in list(hits_g)[:15]: cands_raw[sid].append((cid, 0.65))

    # ── Blocker H: Postal code + House number / DBA (NEW) ────
    logger.info('Blocker H: Postal+house (DBA) index ...')
    postal_idx = _postal_house_index(s23_df)
    for sid, postal, house in zip(s1_df['entity_id'], s1_df['postal_code'], s1_df['house_num']):
        if postal and house:
            key = f'{postal}|{house}'
            for cid in list(postal_idx.get(key, set()))[:20]: cands_raw[sid].append((cid, 1.3))
        elif postal and len(postal) >= 5:
            for cid in list(postal_idx.get(postal, set()))[:10]: cands_raw[sid].append((cid, 0.55))

    # ── Union + dedup + sort by max score → top-K ────────────
    logger.info('Merging 8-blocker union ...')
    final: Dict[str, List[str]] = {}
    for sid in s1_ids:
        score_map: Dict[str, float] = {}
        for cid, sc in cands_raw[sid]:
            score_map[cid] = max(score_map.get(cid, 0.0), sc)
        top = sorted(score_map.items(), key=lambda x: -x[1])[:k]
        final[sid] = [cid for cid, _ in top]

    elapsed = time.time() - t0
    avg_cands = sum(len(v) for v in final.values()) / max(len(s1_ids), 1)
    diag = {'elapsed_sec': elapsed, 'avg_candidates': avg_cands}

    if gt_dict:
        total_gt = sum(len(v) for v in gt_dict.values())
        found_gt = sum(1 for sid, gids in gt_dict.items()
                       for g in gids if g in set(final.get(sid, [])))
        recall = found_gt / total_gt * 100 if total_gt else 0.0
        diag.update({'candidate_recall': recall, 'found_gt': found_gt, 'total_gt': total_gt})
        logger.info(f'Candidate Recall: {recall:.2f}%  ({found_gt}/{total_gt})')

    logger.info(f'Avg cands/S1: {avg_cands:.1f}  |  Elapsed: {elapsed:.1f}s')
    report_memory()
    return final, diag

print('Multi-blocker module loaded (8 routes: A-H).')


### 🧠 Cell 6 · Feature Engineering (36 Features)

In [ ]:
# ===========================================================
# MODULE: Pairwise Feature Extraction (44 features)
# ===========================================================
FEATURE_NAMES = [
    # Name string similarities (6)
    'name_ratio', 'name_token_sort', 'name_token_set',
    'name_partial', 'name_jw', 'name_jaccard',
    # Address string similarities (5)
    'addr_ratio', 'addr_token_sort', 'addr_token_set',
    'addr_jaccard', 'addr_num_overlap',
    # Global
    'country_match',
    # Blocking signal
    'blocking_score',
    # Length ratios (4)
    'name_len_s1', 'name_len_s23', 'name_len_ratio', 'addr_len_ratio',
    # Full document (3)
    'doc_ratio', 'doc_token_set', 'doc_jw',
    # Structural interactions (8)
    'name_x_addr', 'min_name_addr', 'max_name_addr',
    'both_addr', 'missing_asym', 'exact_name',
    'conflict_digits', 'strong_name_num',
    # Hard negative features (5)
    'name_num_overlap', 'name_conflict_digits',
    'addr_jw', 'name_len_diff', 'addr_len_diff',
    # Ranking (2)
    'cand_rank', 'is_s3',
    # === NEW: Phonetic (2) ===
    'phonetic_name_jw', 'phonetic_name_lev',
    # === NEW: Granular address sub-metrics (3) ===
    'postal_exact_match', 'house_num_exact', 'street_name_jaccard',
    # === NEW: Structural asymmetry (1) ===
    'addr_detail_asymmetry',
    # === NEW: DBA / same-location signal (2) ===
    'same_postal_diff_name', 'same_house_same_postal',
    # === NEW: Multilingual Embedding (1) ===
    'embed_cosine',
    # === NEW: Franchise/Conflict Detector (1) ===
    'name_addr_conflict',  # I(name_token_set>=0.85 AND digit_jaccard==0.0)
]
assert len(FEATURE_NAMES) == 46, f'Expected 44, got {len(FEATURE_NAMES)}'

def _ratio(a, b):      return fuzz.ratio(a, b) / 100.0
def _token_sort(a, b): return fuzz.token_sort_ratio(a, b) / 100.0
def _token_set(a, b):  return fuzz.token_set_ratio(a, b) / 100.0
def _partial(a, b):    return fuzz.partial_ratio(a, b) / 100.0
def _jw(a, b):         return JaroWinkler.similarity(a, b)
def _lev_norm(a, b):
    from rapidfuzz.distance import Levenshtein
    ml = max(len(a), len(b))
    return 1.0 - Levenshtein.distance(a, b) / ml if ml > 0 else 1.0

def _jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / max(len(sa | sb), 1)

def _num_set(text): return set(_DIGIT_RE.findall(text))

def _num_overlap(a, b):
    na, nb = _num_set(a), _num_set(b)
    if not na or not nb: return 0.0
    return len(na & nb) / max(len(na | nb), 1)

def _len_ratio(a, b):
    la, lb = len(a), len(b)
    return min(la, lb) / max(la, lb) if max(la, lb) > 0 else 1.0

def _street_tokens(addr: str) -> set:
    """Address tokens with digits stripped — for street name comparison."""
    return {t for t in addr.split() if not t.isdigit() and len(t) > 1}

def compute_features(s1_name, s1_addr, s1_country,
                     s23_name, s23_addr, s23_country,
                     s23_id, cand_rank,
                     s1_postal='', s23_postal='',
                     s1_house='', s23_house='',
                     s1_phonetic='', s23_phonetic='') -> List[float]:
    # ── Name (6) ────────────────────────────────────────────
    f_nr   = _ratio(s1_name, s23_name)
    f_nts  = _token_sort(s1_name, s23_name)
    f_ntse = _token_set(s1_name, s23_name)
    f_np   = _partial(s1_name, s23_name)
    f_njw  = _jw(s1_name, s23_name)
    f_nj   = _jaccard(s1_name, s23_name)
    # ── Address (5) ─────────────────────────────────────────
    f_ar   = _ratio(s1_addr, s23_addr)
    f_ats  = _token_sort(s1_addr, s23_addr)
    f_atse = _token_set(s1_addr, s23_addr)
    f_aj   = _jaccard(s1_addr, s23_addr)
    f_ano  = _num_overlap(s1_addr, s23_addr)
    # ── Global (1) ──────────────────────────────────────────
    f_cm   = 1.0 if (s1_country and s1_country == s23_country) else 0.0
    # ── Blocking (1) ────────────────────────────────────────
    f_bs   = float(cand_rank)
    # ── Length (4) ──────────────────────────────────────────
    f_nls1  = float(len(s1_name))
    f_nls23 = float(len(s23_name))
    f_nlr   = _len_ratio(s1_name, s23_name)
    f_alr   = _len_ratio(s1_addr, s23_addr)
    # ── Full document (3) ────────────────────────────────────
    doc1 = f'{s1_name} {s1_addr} {s1_country}'.strip()
    doc2 = f'{s23_name} {s23_addr} {s23_country}'.strip()
    f_dr  = _ratio(doc1, doc2)
    f_dtse= _token_set(doc1, doc2)
    f_djw = _jw(doc1, doc2)
    # ── Structural interactions (8) ──────────────────────────
    f_nxa  = f_ntse * f_atse
    f_mina = min(f_ntse, f_atse)
    f_maxa = max(f_ntse, f_atse)
    l1, l2 = len(s1_addr), len(s23_addr)
    f_ba   = 1.0 if (l1 >= 5 and l2 >= 5) else 0.0
    f_ma   = 1.0 if ((l1 >= 10 and l2 < 3) or (l2 >= 10 and l1 < 3)) else 0.0
    f_en   = 1.0 if (s1_name and s1_name == s23_name) else 0.0
    d1, d2 = _num_set(s1_addr), _num_set(s23_addr)
    f_cd   = 1.0 if (d1 and d2 and not (d1 & d2)) else 0.0
    f_snn  = 1.0 if (f_ntse >= 0.85 and f_ano >= 0.5) else 0.0
    # ── Hard negative features (5) ──────────────────────────
    nd1, nd2  = _num_set(s1_name), _num_set(s23_name)
    f_nno  = _num_overlap(s1_name, s23_name)
    f_ncd  = 1.0 if (nd1 and nd2 and not (nd1 & nd2)) else 0.0
    f_ajw  = _jw(s1_addr, s23_addr)
    f_nld  = float(abs(len(s1_name) - len(s23_name)))
    f_ald  = float(abs(l1 - l2))
    # ── Ranking (2) ─────────────────────────────────────────
    f_cr   = float(cand_rank)
    f_is3  = 1.0 if s23_id.startswith('S3-') else 0.0
    # ── NEW: Phonetic (2) ───────────────────────────────────
    f_ph_jw  = _jw(s1_phonetic, s23_phonetic) if (s1_phonetic and s23_phonetic) else 0.0
    f_ph_lev = _lev_norm(s1_phonetic, s23_phonetic) if (s1_phonetic and s23_phonetic) else 0.0
    # ── NEW: Granular address sub-metrics (3) ───────────────
    f_post_exact = 1.0 if (s1_postal and s1_postal == s23_postal) else 0.0
    f_house_exact= 1.0 if (s1_house and s1_house == s23_house) else 0.0
    st1, st2 = _street_tokens(s1_addr), _street_tokens(s23_addr)
    f_street_j  = len(st1 & st2) / max(len(st1 | st2), 1)
    # ── NEW: Structural asymmetry (1) ────────────────────────
    s1_rich  = len(s1_addr.split()) >= 4
    s23_rich = len(s23_addr.split()) >= 4
    f_asym   = 1.0 if (s1_rich != s23_rich) else 0.0
    # ── NEW: DBA / same-location (2) ────────────────────────
    # Same postal, very different name → likely DBA/trade name pair
    f_same_post_diff_name = 1.0 if (f_post_exact == 1.0 and f_ntse < 0.5) else 0.0
    f_same_house_post     = 1.0 if (f_post_exact == 1.0 and f_house_exact == 1.0) else 0.0

    return [
        f_nr, f_nts, f_ntse, f_np, f_njw, f_nj,
        f_ar, f_ats, f_atse, f_aj, f_ano,
        f_cm, f_bs,
        f_nls1, f_nls23, f_nlr, f_alr,
        f_dr, f_dtse, f_djw,
        f_nxa, f_mina, f_maxa,
        f_ba, f_ma, f_en, f_cd, f_snn,
        f_nno, f_ncd, f_ajw, f_nld, f_ald,
        f_cr, f_is3,
        # NEW 8
        f_ph_jw, f_ph_lev,
        f_post_exact, f_house_exact, f_street_j,
        f_asym,
        f_same_post_diff_name, f_same_house_post,
        # feature 45 — filled post-hoc if embeddings loaded
        0.0,  # embed_cosine placeholder (replaced by Cell 14)
        # feature 46: name_addr_conflict (franchise detector)
        1.0 if (f_ntse >= 0.85 and f_cd == 1.0) else 0.0,
    ]

def build_lookup(df: pd.DataFrame) -> Dict:
    return {
        r['entity_id']: (
            r['name_norm'], r['addr_norm'], r['country_norm'],
            r.get('postal_code', ''), r.get('house_num', ''), r.get('phonetic_name', '')
        )
        for _, r in df.iterrows()
    }

def build_feature_matrix(pairs, s1_lu, s23_lu, cand_ranks, n_jobs=-1, use_embed=True):
    from joblib import Parallel, delayed
    def _row(s1id, s23id):
        n1, a1, c1, p1, h1, ph1 = s1_lu.get(s1id,  ('','','','','',''))
        n2, a2, c2, p2, h2, ph2 = s23_lu.get(s23id, ('','','','','',''))
        rank = cand_ranks.get((s1id, s23id), 50)
        return compute_features(n1, a1, c1, n2, a2, c2, s23id, rank,
                                p1, p2, h1, h2, ph1, ph2)
    rows = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_row)(s1id, s23id) for s1id, s23id in pairs
    if use_embed and _EMBED_CACHE:
        for idx, (s1id, s23id) in enumerate(pairs):
            X[idx, -1] = embed_cosine(s1id, s23id)
    )
    X = np.array(rows, dtype=np.float32)


# ── Multilingual Embedding Feature (precomputed, cached) ────────────────────
# This is computed ONCE per entity set in the embedding cell below, then
# looked up during feature extraction. We add it as feature #45.
_EMBED_CACHE: dict = {}   # entity_id -> np.ndarray (384-dim)

def get_embed(eid):
    return _EMBED_CACHE.get(eid, None)

def embed_cosine(eid1, eid2):
    """Cached cosine between two entity embeddings."""

    e1, e2 = get_embed(eid1), get_embed(eid2)
    if e1 is None or e2 is None: return 0.0
    dot = float(np.dot(e1, e2))
    return max(0.0, min(1.0, dot))   # embeddings are pre-normalised

print(f'Feature module loaded. Total features: {len(FEATURE_NAMES)} (36 base + 8 new)')


### 📊 Cell 7 · Evaluation Utilities

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODULE: Evaluation — Macro F0.5, Precision, Recall, Singletons
# ═══════════════════════════════════════════════════════════
def _f_beta(p, r, beta=0.5):
    b2 = beta**2
    return (1 + b2) * p * r / (b2 * p + r) if (b2 * p + r) > 0 else 0.0

def entity_score(predicted: List[str], ground_truth: List[str]) -> float:
    gt_set = set(ground_truth)
    pred_set = set(predicted)
    if not gt_set:                    # singleton: 1.0 if no prediction
        return 1.0 if not pred_set else 0.0
    if not pred_set:                  # missed entity
        return 0.0
    tp = len(gt_set & pred_set)
    p  = tp / len(pred_set)
    r  = tp / len(gt_set)
    return _f_beta(p, r)

def evaluate(pred_dict: Dict, gt_dict: Dict) -> Dict:
    """pred_dict: {s1_id: [s23_ids]}  gt_dict: {s1_id: [s23_ids]}"""
    scores, precs, recs = [], [], []
    n_singleton_correct = 0
    n_singletons = 0
    for sid, gt_ids in gt_dict.items():
        preds = pred_dict.get(sid, [])
        gt_set, pred_set = set(gt_ids), set(preds)
        if not gt_set:
            n_singletons += 1
            if not pred_set: n_singleton_correct += 1
        tp = len(gt_set & pred_set)
        p  = tp / len(pred_set) if pred_set else (1.0 if not gt_set else 0.0)
        r  = tp / len(gt_set)   if gt_set   else (1.0 if not pred_set else 0.0)
        scores.append(_f_beta(p, r))
        precs.append(p); recs.append(r)
    n = len(scores)
    return {
        'macro_f05':         float(np.mean(scores)) if n else 0.0,
        'macro_precision':   float(np.mean(precs))  if n else 0.0,
        'macro_recall':      float(np.mean(recs))   if n else 0.0,
        'singleton_acc':     n_singleton_correct / max(n_singletons, 1),
        'n_entities':        n,
    }

def split_three_way(s1_df, train_f=TRAIN_FRAC, val_f=VAL_FRAC, seed=RANDOM_STATE):
    ids = s1_df['entity_id'].tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_tr = int(n * train_f)
    n_va = int(n * val_f)
    tr_set = set(ids[:n_tr])
    va_set = set(ids[n_tr:n_tr+n_va])
    ho_set = set(ids[n_tr+n_va:])
    s1_tr = s1_df[s1_df['entity_id'].isin(tr_set)].reset_index(drop=True)
    s1_va = s1_df[s1_df['entity_id'].isin(va_set)].reset_index(drop=True)
    s1_ho = s1_df[s1_df['entity_id'].isin(ho_set)].reset_index(drop=True)
    return s1_tr, s1_va, s1_ho

print('✅ Evaluation module loaded.')


### 🚀 Cell 8 · TRAINING RUN (Phase 2 gate)
> Set `SAMPLE_S1` and `FULL_S23` in Cell 2 before running this cell.
> For the **Phase 2 gate** run: `SAMPLE_S1=50_000, FULL_S23=True`
> For a **full training run**: `SAMPLE_S1=None, FULL_S23=False`

### 🌍 Cell 14b · Multilingual Embedding Precomputation (MiniLM T4-GPU)

In [ ]:
# ===========================================================
# Precompute entity embeddings: sentence-transformers MiniLM (MIT)
# Model: paraphrase-multilingual-MiniLM-L12-v2 (~120MB, 50 langs)
# Runtime: ~3-5 min on T4 for 50K entities
# ===========================================================
if USE_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer
    import torch

    _embed_ckpt = CKPT_DIR / 'features' / f'embeddings_s1_{SAMPLE_S1}.npy'
    _embed_ids_ckpt = CKPT_DIR / 'features' / f'embed_ids_s1_{SAMPLE_S1}.pkl'

    def _precompute_embeddings(df, model, batch_size=EMBED_BATCH_SIZE):
        texts = [f"{n} {a}".strip() for n, a in zip(df['name_norm'], df['addr_norm'])]
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'  Embedding {len(texts):,} entities on {device}...')
        t0 = time.time()
        embs = model.encode(
            texts, batch_size=batch_size, show_progress_bar=True,
            normalize_embeddings=True, device=device, convert_to_numpy=True
        )
        print(f'  Done in {time.time()-t0:.1f}s. Shape: {embs.shape}')
        return embs

    print('Loading MiniLM model...')
    _embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    print('Model loaded.')

    # Compute / load for all entities (S1 + S23)
    all_df = pd.concat([s1, pd.concat([s2, s3], ignore_index=True)], ignore_index=True)
    ckpt_key = f'embed_{SAMPLE_S1}_{FULL_S23}'
    _ckpt_f = CKPT_DIR / 'features' / f'{ckpt_key}.npy'
    _ckpt_ids = CKPT_DIR / 'features' / f'{ckpt_key}_ids.pkl'

    if _ckpt_f.exists() and _ckpt_ids.exists():
        print('Loading embeddings from checkpoint...')
        _embs = np.load(_ckpt_f)
        _ids  = joblib.load(_ckpt_ids)
    else:
        _embs = _precompute_embeddings(all_df, _embed_model)
        _ids  = list(all_df['entity_id'])
        np.save(_ckpt_f, _embs)
        joblib.dump(_ids, _ckpt_ids)
        save_to_drive(_ckpt_f)
        print('Embeddings saved to checkpoint.')

    # Populate global cache
    _EMBED_CACHE.clear()
    for eid, emb in zip(_ids, _embs):
        _EMBED_CACHE[eid] = emb.astype(np.float32)

    del _embed_model, _embs; free_gpu()
    print(f'Embedding cache loaded: {len(_EMBED_CACHE):,} entities.')
    report_memory()
else:
    print('Embeddings disabled (USE_EMBEDDINGS=False). Feature 45 will be 0.')


In [ ]:
print(f'=== TRAINING RUN | S1_sample={SAMPLE_S1} | full_s23={FULL_S23} | K={K_CANDIDATES} ===')

# ── Step 1: Load Data ────────────────────────────────────────
print('\n[1/6] Loading data …')
s1, s2, s3, gt = load_data('train', sample_s1=SAMPLE_S1, full_s23=FULL_S23)
print(f'  S1={len(s1):,}  S2={len(s2):,}  S3={len(s3):,}  GT_entries={len(gt):,}')

# ── Step 2: 3-Way Split ──────────────────────────────────────
print('\n[2/6] Splitting Train / Val / Holdout …')
s1_tr, s1_va, s1_ho = split_three_way(s1)
print(f'  Train={len(s1_tr):,}  Val={len(s1_va):,}  Holdout={len(s1_ho):,}')
gt_tr = {k: v for k, v in gt.items() if k in set(s1_tr['entity_id'])}
gt_va = {k: v for k, v in gt.items() if k in set(s1_va['entity_id'])}
gt_ho = {k: v for k, v in gt.items() if k in set(s1_ho['entity_id'])}

# ── Step 3: Multi-Blocker Candidate Generation ───────────────
print('\n[3/6] Generating candidates (all 6 blockers) …')
t_block = time.time()
candidates, block_diag = generate_candidates(s1, s2, s3, k=K_CANDIDATES, gt_dict=gt)
print(f'  Blocking done in {time.time()-t_block:.1f}s')
print(f'  ★ Candidate Recall : {block_diag.get("candidate_recall", "N/A"):.2f}%')
print(f'  ★ Avg Cands / S1   : {block_diag["avg_candidates"]:.1f}')

# ── Step 4: Build Feature Matrices ──────────────────────────
print('\n[4/6] Building feature matrices …')
s23_all = pd.concat([s2, s3], ignore_index=True)
s23_lu = build_lookup(s23_all)

def _pairs_labels_with_hard_negs(s1_sub, cands, gt_sub, neg_ratio=5):
    """
    HONEST pairs + targeted hard-negative oversampling.
    Hard negatives: cand_pairs where name_token_set >= 0.85 AND country_match
    AND ground truth == 0. These are the pairs that break the model.
    neg_ratio: hard negatives per positive in final batch.
    """
    positives, hard_negs, easy_negs = [], [], []
    for sid in s1_sub['entity_id']:
        gt_set = set(gt_sub.get(sid, []))
        s1_info = s1_lu.get(sid, ('','','','','',''))
        for cid in cands.get(sid, []):
            label = 1 if cid in gt_set else 0
            if label == 1:
                positives.append((sid, cid, 1))
            else:
                s23_info = s23_lu.get(cid, ('','','','','',''))
                name_sim = fuzz.token_set_ratio(s1_info[0], s23_info[0]) / 100.0
                same_country = (s1_info[2] and s1_info[2] == s23_info[2])
                if name_sim >= 0.85 and same_country:
                    hard_negs.append((sid, cid, 0))
                else:
                    easy_negs.append((sid, cid, 0))

    n_pos = len(positives)
    n_hard_target = min(n_pos * neg_ratio, len(hard_negs))
    n_easy_target = max(0, n_pos * neg_ratio - n_hard_target)

    rng = np.random.default_rng(RANDOM_STATE)
    if n_hard_target < len(hard_negs):
        idx = rng.choice(len(hard_negs), n_hard_target, replace=False)
        hard_negs = [hard_negs[i] for i in idx]
    if n_easy_target < len(easy_negs):
        idx = rng.choice(len(easy_negs), n_easy_target, replace=False)
        easy_negs = [easy_negs[i] for i in idx]

    all_pairs = positives + hard_negs + easy_negs
    rng.shuffle(all_pairs)
    pairs  = [(s, c) for s, c, _ in all_pairs]
    labels = [lbl for _, _, lbl in all_pairs]
    print(f'  Pairs: {len(pairs):,}  (pos={n_pos:,}  hard_neg={len(hard_negs):,}  easy_neg={len(easy_negs):,})')
    return pairs, labels

cands_tr = {k: v for k, v in candidates.items() if k in set(s1_tr['entity_id'])}
cands_va = {k: v for k, v in candidates.items() if k in set(s1_va['entity_id'])}
cands_ho = {k: v for k, v in candidates.items() if k in set(s1_ho['entity_id'])}

pairs_tr, labels_tr = _pairs_labels_with_hard_negs(s1_tr, cands_tr, gt_tr)
pairs_va, labels_va = _pairs_labels(s1_va, cands_va, gt_va)
pairs_ho, labels_ho = _pairs_labels(s1_ho, cands_ho, gt_ho)

ranks = {(sid, cid): r+1 for sid, clist in candidates.items() for r, cid in enumerate(clist)}
s1_lu = build_lookup(s1)

print(f'  Train pairs: {len(pairs_tr):,}  (pos={sum(labels_tr):,})')
X_tr = build_feature_matrix(pairs_tr, s1_lu, s23_lu, ranks, n_jobs=-1)
X_va = build_feature_matrix(pairs_va, s1_lu, s23_lu, ranks, n_jobs=-1)
X_ho = build_feature_matrix(pairs_ho, s1_lu, s23_lu, ranks, n_jobs=-1)
    pairs_ho_list = [(s, c) for s, c in pairs_ho]
    feat_map_ho   = {(s,c): dict(zip(FEATURE_NAMES, X_ho[k].tolist())) for k,(s,c) in enumerate(pairs_ho_list)}
y_tr = np.array(labels_tr, dtype=np.int32)
y_va = np.array(labels_va, dtype=np.int32)
y_ho = np.array(labels_ho, dtype=np.int32)
print(f'  X_tr: {X_tr.shape}  X_va: {X_va.shape}  X_ho: {X_ho.shape}')

# ── Step 5: Train LightGBM ───────────────────────────────────
print('\n[5/6] Training LightGBM …')
    print('\n[5/6] Training Ensemble (LightGBM + CatBoost + XGBoost) ...')
    models = train_ensemble(X_tr, y_tr, X_va, y_va)
    params, dtrain, num_boost_round=LGBM_PARAMS['n_estimators'],
    valid_sets=[dval],
)

# ── Step 6: Threshold Sweep on Val ──────────────────────────
print('\n[6/6] Threshold sweep on validation …')
va_probs = model.predict(X_va)
prob_map_va = {(sid, cid): p for (sid, cid), p in zip(pairs_va, va_probs)}

best_f05, best_thr = -1.0, 0.5
thr_table = []
for thr in THRESHOLD_GRID:
    pred_va = {sid: [c for c in cands_va.get(sid, []) if prob_map_va.get((sid, c), 0) >= thr]
               for sid in s1_va['entity_id']}
    stats = evaluate(pred_va, gt_va)
    thr_table.append({'threshold': thr, **stats})
    if stats['macro_f05'] > best_f05:
        best_f05, best_thr = stats['macro_f05'], thr

thr_df = pd.DataFrame(thr_table)
print(thr_df.to_string(index=False, float_format='{:.4f}'.format))
print(f'\n  ★ Best Threshold: {best_thr:.2f}  |  Val F0.5: {best_f05:.4f}')

# Holdout evaluation
    ho_probs = ensemble_predict(models, X_ho)
prob_map_ho = {(sid, cid): p for (sid, cid), p in zip(pairs_ho, ho_probs)}
pred_ho = {sid: [c for c in cands_ho.get(sid, []) if prob_map_ho.get((sid, c), 0) >= best_thr]
           for sid in s1_ho['entity_id']}
ho_stats = evaluate(pred_ho, gt_ho)
print(f'  ★ Holdout F0.5   : {ho_stats["macro_f05"]:.4f}  (threshold={best_thr:.2f})')

# Save checkpoints
    joblib.dump(models, MODEL_DIR/'ensemble_models.joblib')
joblib.dump({'threshold': best_thr, 'val_f05': best_f05, 'ho_stats': ho_stats}, MODEL_DIR/'meta.joblib')
                                           'best_val_stats': best_val_stats, 
'ho_stats': ho_stats,
print('\n✅ Model and metadata saved to', MODEL_DIR)

# Export threshold + metrics to JSON
import json as _json
_summary = {
    'best_threshold': best_thr,
    'val_f05': best_f05,
    'val_precision': best_val_stats.get('macro_precision', 0.0),
    'val_recall': best_val_stats.get('macro_recall', 0.0),
    'holdout_f05': ho_stats['macro_f05'],
    'holdout_precision': ho_stats.get('macro_precision', 0.0),
    'n_features': len(FEATURE_NAMES),
    'sample_s1': SAMPLE_S1,
    'k_candidates': K_CANDIDATES,
    'candidate_recall': block_diag.get('candidate_recall', None),
}
_json_path = OUTPUT_DIR / 'experiment_summary.json'
with open(_json_path, 'w') as _f:
    _json.dump(_summary, _f, indent=2)
print(f'  Experiment summary saved -> {_json_path}')
print(_json.dumps(_summary, indent=2))


### 🤖 Cell 16b · Ensemble Training (LightGBM + CatBoost + XGBoost)
**Geometric mean blending:** `Score = P_lgbm^0.5 × P_catboost^0.3 × P_xgb^0.2`

In [ ]:
# ===========================================================
# MODULE: Ensemble Trainer
# LightGBM (0.5) + CatBoost (0.3) + XGBoost (0.2)
# ===========================================================
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

def train_ensemble(X_tr, y_tr, X_va, y_va):
    models = {}

    # ── LightGBM ─────────────────────────────────────────────
    print('  [1/3] Training LightGBM...')
    params_lgb = {k: v for k, v in LGBM_PARAMS.items() if k != 'n_estimators'}
    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURE_NAMES)
    dval   = lgb.Dataset(X_va, label=y_va, feature_name=FEATURE_NAMES, reference=dtrain)
    models['lgbm'] = lgb.train(
        params_lgb, dtrain, num_boost_round=LGBM_PARAMS['n_estimators'],
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                   lgb.log_evaluation(200)],
    )
    print(f'  LightGBM: {models["lgbm"].best_iteration} trees')

    # ── CatBoost ─────────────────────────────────────────────
    print('  [2/3] Training CatBoost...')
    cb = CatBoostClassifier(
        iterations=1000, learning_rate=0.05, depth=8,
        l2_leaf_reg=3, border_count=128, random_strength=1,
        eval_metric='AUC', use_best_model=True,
        early_stopping_rounds=50, verbose=200,
        task_type='CPU', random_seed=RANDOM_STATE,
    )
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), feature_names=FEATURE_NAMES)
    models['catboost'] = cb
    print(f'  CatBoost: {cb.best_iteration_} trees')

    # ── XGBoost ──────────────────────────────────────────────
    print('  [3/3] Training XGBoost...')
    xgb_params = dict(
        objective='binary:logistic', eval_metric='auc',
        max_depth=8, learning_rate=0.05, n_estimators=1000,
        subsample=0.80, colsample_bytree=0.80,
        reg_alpha=0.1, reg_lambda=0.1,
        n_jobs=-1, random_state=RANDOM_STATE,
        tree_method='hist', early_stopping_rounds=50,
    )
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=200)
    models['xgb'] = xgb_model
    print(f'  XGBoost: {xgb_model.best_iteration} trees')

    return models

def ensemble_predict(models, X):
    """Geometric mean blend: P_lgbm^0.5 * P_cb^0.3 * P_xgb^0.2"""
    w = ENSEMBLE_WEIGHTS
    p_lgbm = np.clip(models['lgbm'].predict(X), 1e-7, 1-1e-7)
    p_cb   = np.clip(models['catboost'].predict_proba(X)[:, 1], 1e-7, 1-1e-7)
    p_xgb  = np.clip(models['xgb'].predict_proba(X)[:, 1], 1e-7, 1-1e-7)
    return p_lgbm**w['lgbm'] * p_cb**w['catboost'] * p_xgb**w['xgb']

print('Ensemble training module loaded (LightGBM + CatBoost + XGBoost).')


### 🔬 Cell 9 · Missed-Match Analysis (Phase 3)
> Run this **after Cell 8** to categorize what K=50 misses.
> Output: `reports/error_analysis/missed_blocking_matches.csv`

### 🔺 Cell 17 · Post-Processing: Triangulation + Singleton Guardrail
**Two strategies:**
1. **S2↔S3 Triangulation** — if S2a strongly matches S3b, and S1→S2a is marginal, boost both via graph connectivity
2. **Singleton Guardrail** — lone predictions below `T_SINGLE` are pruned to protect 1.0 singleton scores

In [ ]:
# ===========================================================
# MODULE: Post-Processing (Production Grade)
# 1. NetworkX graph transitive closure + S2↔S3 triangulation
# 2. Feature-aware singleton precision shield
# ===========================================================
import networkx as nx

# ── S2↔S3 cross-similarity index (TF-IDF, built once per session) ──────────
_S2_S3_IDX: Optional[dict] = None

def _build_s2_s3_index(s2_df, s3_df, threshold=0.85):
    """
    Build a TF-IDF cosine cross-similarity lookup: {s2_id: [s3_ids >= threshold]}.
    Used by the triangulation engine to confirm marginal S1→S2 predictions.
    """
    all_texts = list(s2_df['blocking_text']) + list(s3_df['blocking_text'])
    vec = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER, ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES, min_df=TFIDF_MIN_DF,
        sublinear_tf=True, norm='l2', dtype=np.float32
    )
    vec.fit(all_texts)
    s2_mat = vec.transform(list(s2_df['blocking_text']))
    s3_mat = vec.transform(list(s3_df['blocking_text']))
    s2_ids = list(s2_df['entity_id'])
    s3_ids = list(s3_df['entity_id'])
    K_CROSS = 5
    result = defaultdict(list)
    for start in range(0, len(s2_ids), S1_BATCH_SIZE):
        batch = s2_mat[start:start+S1_BATCH_SIZE].toarray().astype(np.float32)
        scores = (s3_mat @ batch.T).toarray()  # (N_s3, batch)
        for bi in range(scores.shape[1]):
            row = scores[:, bi]
            top_idx = np.argpartition(row, -min(K_CROSS, len(row)))[-K_CROSS:]
            for idx in top_idx:
                if row[idx] >= threshold:
                    result[s2_ids[start+bi]].append(s3_ids[idx])
    free_gpu()
    return dict(result)


def build_triangulated_matches(
        s1_ids: list,
        predicted_pairs: dict,
        prob_map: dict,
        s23_lu: dict,
        s2_df, s3_df,
        t_high: float = T_HIGH,
        t_support: float = 0.40):
    """
    NetworkX graph transitive closure + S2↔S3 triangulation.

    Algorithm:
    1. Build undirected graph G where nodes = entity IDs.
       Add edge (S1_i, S23_j) for every prediction with prob >= t_high.
    2. For each S1, cross-evaluate its S2 and S3 candidates pairwise:
       - If S2a and S3b are near-twins (name_sim >= 0.88),
         accept both if either has prob >= t_support.
    3. Compute connected components of G — pull any S23 node reachable
       from an S1 node into that S1's match set.
    """
    global _S2_S3_IDX
    if _S2_S3_IDX is None:
        print('  Building S2<->S3 TF-IDF cross-index...')
        _S2_S3_IDX = _build_s2_s3_index(s2_df, s3_df, threshold=0.85)
        print(f'  S2->S3 index: {len(_S2_S3_IDX):,} S2 nodes with >=1 S3 twin')

    G = nx.Graph()
    n_graph_edges = 0
    n_cross_boosted = 0

    for s1_id in s1_ids:
        G.add_node(s1_id)
        cands = predicted_pairs.get(s1_id, [])
        if not cands:
            continue

        s2_cands = [c for c in cands if c.startswith('S2-')]
        s3_cands = [c for c in cands if c.startswith('S3-')]

        # Step 1: Add high-confidence edges to graph
        for cid in cands:
            p = prob_map.get((s1_id, cid), 0.0)
            if p >= t_high:
                G.add_edge(s1_id, cid, weight=p)
                n_graph_edges += 1

        # Step 2: S2↔S3 cross-evaluation
        for s2_id in s2_cands:
            p_s2 = prob_map.get((s1_id, s2_id), 0.0)
            for s3_id in (_S2_S3_IDX.get(s2_id, []) + s3_cands):
                p_s3 = prob_map.get((s1_id, s3_id), 0.0)
                if p_s2 < t_support and p_s3 < t_support:
                    continue
                # Direct S2↔S3 name similarity
                n2, a2 = (s23_lu.get(s2_id, ('','','','','',''))[:2])
                n3, a3 = (s23_lu.get(s3_id, ('','','','','',''))[:2])
                name_sim = fuzz.token_set_ratio(n2, n3) / 100.0
                addr_sim = fuzz.token_set_ratio(a2, a3) / 100.0 if (a2 and a3) else 1.0
                if name_sim >= 0.88 and addr_sim >= 0.70:
                    # Both are near-twins → add to graph as mutually confirmed
                    G.add_edge(s2_id, s3_id, weight=name_sim * addr_sim)
                    if p_s2 >= t_support:
                        G.add_edge(s1_id, s2_id, weight=p_s2)
                        n_cross_boosted += 1
                    if p_s3 >= t_support:
                        G.add_edge(s1_id, s3_id, weight=p_s3)
                        n_cross_boosted += 1

        # Step 3: Pull S2-twins of high-confidence S2s (TF-IDF index)
        for s2_id in s2_cands:
            if prob_map.get((s1_id, s2_id), 0.0) >= t_high:
                for s3_twin in _S2_S3_IDX.get(s2_id, []):
                    if s3_twin not in (set(cands)):
                        # S3 twin never in original candidates — graph-pull it in
                        G.add_edge(s1_id, s3_twin, weight=0.0)  # pulled by transitivity

    # Step 4: Transitive closure — for each connected component containing an S1,
    # all S23 nodes in that component are included in S1's match set.
    refined: dict = {sid: [] for sid in s1_ids}
    for component in nx.connected_components(G):
        s1_nodes  = [n for n in component if n.startswith('S1-')]
        s23_nodes = [n for n in component if not n.startswith('S1-')]
        for s1_id in s1_nodes:
            refined[s1_id].extend(s23_nodes)

    print(f'  Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
    print(f'  Cross-boosted: {n_cross_boosted:,} marginal pairs confirmed by triangulation')
    return refined


def apply_singleton_guardrail(
        pred_dict: dict,
        prob_map: dict,
        features_map: dict,
        t_single: float = T_SINGLE):
    """
    Feature-aware Singleton Precision Shield.

    Math: For a true singleton, predicting empty -> F0.5 = 1.0.
          A single false positive -> F0.5 = 0.0 (beta=0.5, precision-heavy).
          We prune when EITHER:
          (a) prob < T_SINGLE  — model is not confident enough
          (b) conflict_digits == 1.0 — address digit conflict (franchise risk)
          (c) name_ratio < 0.70 — raw character similarity too weak

    Only applied when len(predictions) == 1 (the truly risky case).
    """
    pruned = 0
    shielded = {}
    for sid, preds in pred_dict.items():
        if len(preds) == 1:
            cid = preds[0]
            p    = prob_map.get((sid, cid), 0.0)
            feat = features_map.get((sid, cid), {})
            has_digit_conflict = feat.get('conflict_digits', 0.0) == 1.0
            low_char_sim       = feat.get('name_ratio', 1.0) < 0.70
            if p < t_single or has_digit_conflict or low_char_sim:
                shielded[sid] = []
                pruned += 1
                continue
        shielded[sid] = preds
    print(f'  Singleton guardrail: pruned {pruned:,} lone marginal predictions -> empty (protected 1.0 score)')
    return shielded


def full_postprocess(
        s1_ids, pred_dict, prob_map, features_map,
        cands, s23_lu, s2_df, s3_df,
        do_triangulation=True, do_guardrail=True):
    """
    Unified post-processor:
    1. NetworkX graph triangulation (recovers missed cross-source matches)
    2. Feature-aware singleton guardrail (precision protection)
    """
    result = dict(pred_dict)
    if do_triangulation:
        print('  Running graph triangulation...')
        result = build_triangulated_matches(s1_ids, result, prob_map, s23_lu, s2_df, s3_df)
    if do_guardrail:
        result = apply_singleton_guardrail(result, prob_map, features_map)
    return result

print('Post-processing module loaded (networkx graph triangulation + feature-aware singleton shield).')


In [ ]:
# Find GT pairs not in any candidate set
missed_rows = []
for sid, gt_ids in gt.items():
    cand_set = set(candidates.get(sid, []))
    for gid in gt_ids:
        if gid not in cand_set:
            s1_info  = s1_lu.get(sid, ('', '', ''))
            s23_info = s23_lu.get(gid, ('', '', ''))
            missed_rows.append({
                's1_id': sid, 'true_id': gid,
                'source': 'S2' if gid.startswith('S2-') else 'S3',
                'name_s1': s1_info[0], 'name_true': s23_info[0],
                'addr_s1': s1_info[1], 'addr_true': s23_info[1],
                'country': s1_info[2],
            })

missed_df = pd.DataFrame(missed_rows)
out_path = REPORT_DIR / 'missed_blocking_matches.csv'
missed_df.to_csv(out_path, index=False)
print(f'✅ Saved {len(missed_df):,} missed GT pairs → {out_path}')
print(f'   Missed from S2: {len(missed_df[missed_df["source"]=="S2"]):,}')
print(f'   Missed from S3: {len(missed_df[missed_df["source"]=="S3"]):,}')
if len(missed_df) > 0:
    print('\n  Sample missed matches:')
    print(missed_df.head(10).to_string(index=False))
    # Quick heuristic categorisation
    def _cat(row):
        n1, n2 = row['name_s1'], row['name_true']
        if not n1 or not n2: return 'MISSING_FIELD'
        if n1 == n2: return 'ADDRESS_VARIATION'
        if fuzz.ratio(n1, n2) > 90: return 'NAME_TYPO'
        if fuzz.token_set_ratio(n1, n2) > 90: return 'WORD_ORDER'
        if any(a in n2.split() or a in n1.split() for a in n1.split()[:2]): return 'ABBREVIATION'
        return 'OTHER'
    missed_df['category'] = missed_df.apply(_cat, axis=1)
    print('\n  Failure category breakdown:')
    print(missed_df['category'].value_counts().to_string())


### 🎯 Cell 10 · Full Test Inference
> Run only after finalising threshold in Cell 8.
> Generates the two required submission files.

In [ ]:
print('=== FULL TEST INFERENCE ===')

# Load saved model + threshold
    models_inf = joblib.load(MODEL_DIR/'ensemble_models.joblib')
meta_inf  = joblib.load(MODEL_DIR/'meta.joblib')
thr_inf   = meta_inf['threshold']
print(f'  Loaded model  (thr={thr_inf:.2f}  val_F0.5={meta_inf["val_f05"]:.4f})')

# Load and preprocess test data
print('  Loading test data …')
s1_test, s2_test, s3_test, _ = load_data('test', sample_s1=None, full_s23=True)
print(f'  Test: S1={len(s1_test):,}  S2={len(s2_test):,}  S3={len(s3_test):,}')

# Block test set
print('  Generating test candidates …')
cands_test, _ = generate_candidates(s1_test, s2_test, s3_test, k=K_CANDIDATES)

# Save candidate_pairs.tsv
cand_rows = [{'source1_entity_id': sid, 'candidate_entity_ids': ','.join(cands_test.get(sid, []))}
             for sid in s1_test['entity_id']]
pd.DataFrame(cand_rows).to_csv(OUTPUT_DIR/'candidate_pairs.tsv', sep='\t', index=False)
print(f'  ✔ candidate_pairs.tsv saved')

# Build features and predict in batches
test_pairs = [(sid, cid) for sid, clist in cands_test.items() for cid in clist]
test_ranks = {(sid, cid): r+1 for sid, clist in cands_test.items() for r, cid in enumerate(clist)}
s23_test_lu = build_lookup(pd.concat([s2_test, s3_test], ignore_index=True))
s1_test_lu  = build_lookup(s1_test)

print(f'  Computing features for {len(test_pairs):,} pairs …')
BATCH = 500_000
all_probs = []
for i in range(0, len(test_pairs), BATCH):
    batch_pairs = test_pairs[i:i+BATCH]
    X_batch = build_feature_matrix(batch_pairs, s1_test_lu, s23_test_lu, test_ranks, n_jobs=-1)
    all_probs.extend(models_inf.predict(X_batch).tolist())
    print(f'    Batch {i//BATCH+1}: {len(batch_pairs):,} pairs processed', end='\r')

prob_map_test = {pair: p for pair, p in zip(test_pairs, all_probs)}

# Apply threshold and write matching_results.tsv
match_rows = []
for sid in s1_test['entity_id']:
    clist   = cands_test.get(sid, [])
    matched = [c for c in clist if prob_map_test.get((sid, c), 0.0) >= thr_inf]
    match_rows.append({'source1_entity_id': sid, 'matched_entity_ids': ','.join(matched)})

    # Apply triangulation + singleton guardrail
    pred_map = {r['source1_entity_id']: r['matched_entity_ids'].split(',') if r['matched_entity_ids'] else [] for r in match_rows}
    feat_map_test = {(r['source1_entity_id'], mid): {} for r in match_rows for mid in (r['matched_entity_ids'].split(',') if r['matched_entity_ids'] else [])}
    pred_map = full_postprocess(list(pred_map.keys()), pred_map, prob_map_test, feat_map_test, cands_test, s23_lu_test, s2_test, s3_test)
    match_rows = [{'source1_entity_id': sid, 'matched_entity_ids': ','.join(preds)} for sid, preds in pred_map.items()]
pd.DataFrame(match_rows).to_csv(OUTPUT_DIR/'matching_results.tsv', sep='\t', index=False)
print(f'\n  ✔ matching_results.tsv saved')
n_matched = sum(1 for r in match_rows if r['matched_entity_ids'])
print(f'  Entities with ≥1 match: {n_matched:,} / {len(match_rows):,}')


### ✅ Cell 11 · Official Submission Validator

In [ ]:
val_script = ROOT / 'utils' / 'validate_submission.py'
if val_script.exists():
    cmd = [sys.executable, str(val_script),
           '--matching',  str(OUTPUT_DIR/'matching_results.tsv'),
           '--candidate', str(OUTPUT_DIR/'candidate_pairs.tsv'),
           '--test-dir',  str(TEST_DIR)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode == 0:
        print('\n=========================================')
        print('  ✅  SUBMISSION VALIDATION  →  PASS  ')
        print('=========================================')
    else:
        print(res.stderr)
        print('\n✗ Validation FAILED — fix errors above before submitting.')
else:
    print(f'⚠ Validator script not found at {val_script}')
    print('  Manually verify output files in:', OUTPUT_DIR)
